# Watershed reconciliation: CPDMI F1 comparison

This notebook measures the effect of the experimental nucleus-seeded watershed reconciliation on the **same public InstanSeg model predictions**. It compares:

1. **Unresolved:** independent nuclear and cell heads (`resolve_cell_and_nucleus=False`).
2. **Native:** InstanSeg's standard biological resolver (`resolve_cell_and_nucleus=True`).
3. **Watershed:** the local nucleus-seeded watershed applied to the unresolved output.

All methods use the same CPDMI records, rescaled images, channels, inference tile size, model postprocessing, and object-matching metric. The main result is cell F1 at IoU 0.5 and the F1-versus-IoU-threshold curve. Nuclear F1 is reported separately; unresolved and watershed nuclei are intentionally identical, so any watershed effect should appear in the cell score.

The default run evaluates the 30-record CPDMI `Validation` split. The saved `Test` split can be enabled below, but in the current combined dataset it is TissueNet rather than a CPDMI held-out test set.

## 1. Imports, paths, and controls

In [ ]:
import gc
import json
import os
import sys
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries, watershed
import torch
from torch import nn

TRAINING_ROOT = Path(os.environ.get(
    'INSTANSEG_TRAINING_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg',
)).expanduser().resolve()
DATASET_DIR = TRAINING_ROOT / 'datasets'
DATASET_FILE = DATASET_DIR / 'segmentation_dataset.pth'
os.environ['INSTANSEG_DATASET_PATH'] = str(DATASET_DIR)
SOURCE_ROOT = Path(os.environ.get(
    'INSTANSEG_EVAL_SOURCE_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/instanseg_multihead_0325_20260825/source/instanseg',
)).expanduser().resolve()
PUBLIC_MODEL_CACHE_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_MODEL_CACHE',
    str(TRAINING_ROOT / 'public_model_cache'),
)).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_WATERSHED_F1_RESULTS',
    str(TRAINING_ROOT / 'model_comparisons' / 'watershed_f1_cpdmi_validation'),
)).expanduser().resolve()

PUBLIC_MODEL_NAME = 'fluorescence_nuclei_and_cells'
PUBLIC_MODEL_VERSION = '0.1.1'
DEVICE = os.environ.get('INSTANSEG_WATERSHED_F1_DEVICE', 'cuda:0')
INFERENCE_TILE_SIZE = 256
INFERENCE_BATCH_SIZE = 1
INFERENCE_DIMENSION_MULTIPLE = 16

VALIDATION_SPLIT = 'Validation'
VALIDATION_PARENT_DATASET = 'CPDMI_2023'
VALIDATION_RECORD_LIMIT = None
INCLUDE_SAVED_TEST = False
TEST_SPLIT = 'Test'
TEST_PARENT_DATASET = None
TEST_RECORD_LIMIT = 100
EXPECTED_CPDMI_VALIDATION_COUNT = 30
RUN_EVALUATION = True
SAVE_ARTIFACTS = True
SPOT_CHECK_RECORDS = 1

# These values are shared by unresolved and native inference. The only changed
# model argument is resolve_cell_and_nucleus.
POSTPROCESSING = {
    'mask_threshold': 0.53,
    'peak_distance': 4,
    'seed_threshold': 0.5,
    'overlap_threshold': 0.5,
    'mean_threshold': -10000.0,
    'window_size': 128,
    'min_size': 10,
    'cleanup_fragments': False,
    'max_seeds': 2000,
}
WATERSHED_OVERLAP_FRACTION = 0.5
THRESHOLDS = [round(float(x), 2) for x in np.linspace(0.5, 1.0, 11)]
METHODS = ('unresolved', 'native', 'watershed')
METHOD_LABELS = {
    'unresolved': 'Unresolved dual heads',
    'native': 'Native InstanSeg resolve',
    'watershed': 'Custom watershed resolve',
}

if not torch.cuda.is_available():
    raise RuntimeError(
        'This notebook is intended for a GPU kernel. The public model has previously '
        'crashed during CPU evaluation; select CUDA before running inference.'
    )
print({'device': DEVICE, 'tile_size': INFERENCE_TILE_SIZE, 'results': str(RESULTS_ROOT)})

## 2. Load the public model and the CPDMI records

In [ ]:
if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(f'Expected InstanSeg source at {SOURCE_ROOT / "instanseg"}')
if not DATASET_FILE.is_file():
    raise FileNotFoundError(DATASET_FILE)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import instanseg
from instanseg import InstanSeg as PublicInstanSeg
from instanseg.utils.augmentations import Augmentations
from instanseg.utils.data_loader import get_image
from instanseg.utils.loss import instanseg_loss as loss_utils
from instanseg.utils.loss.instanseg_loss import InstanSeg as LossInstanSeg
from instanseg.utils.metrics import matching_torch
from instanseg.utils.tiling import _instanseg_padding, _recover_padding, _sliding_window_inference

from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide

model_index = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
entries = [entry for entry in json.loads(model_index.read_text()) if entry.get('name') == PUBLIC_MODEL_NAME]
if not entries or entries[0].get('version') != PUBLIC_MODEL_VERSION:
    raise RuntimeError(f'Expected {PUBLIC_MODEL_NAME} v{PUBLIC_MODEL_VERSION}')
PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)
runner = PublicInstanSeg(model_type=PUBLIC_MODEL_NAME, device=DEVICE, verbosity=0, channels_last=False)
network = runner.instanseg.eval()
MODEL_PIXEL_SIZE_UM = float(network.pixel_size)
if abs(MODEL_PIXEL_SIZE_UM - 0.5) > 1e-6:
    raise ValueError(f'Expected public model pixel size 0.5, got {MODEL_PIXEL_SIZE_UM}')

print({
    'instanseg_source': str(Path(instanseg.__file__).resolve()),
    'model': PUBLIC_MODEL_NAME,
    'version': PUBLIC_MODEL_VERSION,
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
    'cells_and_nuclei': bool(network.cells_and_nuclei),
})

def _safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def _resolve_array(value):
    if isinstance(value, (str, Path)):
        return np.asarray(get_image(str(value)))
    return np.asarray(value)

def _nucleus_value(item):
    return item.get('nucleus_masks') if item.get('nucleus_masks') is not None else item.get('masks')

def _cell_value(item):
    return item.get('cell_masks')

def _record_id(split, index, item):
    return f'{split}[{index}]::{item.get("parent_dataset", "<missing>")}::{item.get("filename", index)}'

def _record_info(split, index, item):
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    reference = nucleus if nucleus is not None else cell
    reference_shape = _resolve_array(reference).shape
    return {
        'split': split,
        'source_index': int(index),
        'record_id': _record_id(split, index, item),
        'filename': str(item.get('filename', index)),
        'parent_dataset': str(item.get('parent_dataset', '')),
        'platform': str(item.get('platform', '')),
        'native_pixel_size_um': float(item['pixel_size']),
        'image_shape': tuple(int(v) for v in reference_shape),
        'nucleus_annotation': nucleus is not None,
        'cell_annotation': cell is not None,
    }

def select_records(dataset, split, parent_dataset=None, limit=None):
    if split not in dataset:
        raise KeyError(f'Missing split {split!r}; available splits: {sorted(dataset)}')
    selected = []
    for index, item in enumerate(dataset[split]):
        if parent_dataset is not None and item.get('parent_dataset') != parent_dataset:
            continue
        if item.get('duplicate', False):
            continue
        if _nucleus_value(item) is None and _cell_value(item) is None:
            continue
        selected.append((index, item))
    if limit is not None:
        selected = selected[:int(limit)]
    return [(_record_info(split, index, item), item) for index, item in selected]

dataset = _safe_torch_load(DATASET_FILE)
validation_records = select_records(
    dataset, VALIDATION_SPLIT, VALIDATION_PARENT_DATASET, VALIDATION_RECORD_LIMIT
)
if VALIDATION_RECORD_LIMIT is None and len(validation_records) != EXPECTED_CPDMI_VALIDATION_COUNT:
    raise RuntimeError(f'Expected {EXPECTED_CPDMI_VALIDATION_COUNT} CPDMI validation records, found {len(validation_records)}')

RECORD_GROUPS = {'CPDMI validation': validation_records}
if INCLUDE_SAVED_TEST:
    RECORD_GROUPS['Saved Test split'] = select_records(
        dataset, TEST_SPLIT, TEST_PARENT_DATASET, TEST_RECORD_LIMIT
    )
RECORDS = [record for group in RECORD_GROUPS.values() for record in group]
selection_table = pd.DataFrame([info for info, _ in RECORDS])
display(selection_table.head())
print('Records by evaluation set:', {name: len(items) for name, items in RECORD_GROUPS.items()})
if INCLUDE_SAVED_TEST:
    print('Saved Test parent datasets:', selection_table.loc[selection_table['split'] == TEST_SPLIT, 'parent_dataset'].value_counts().to_dict())

## 3. Prepare every record once at the public model's scale

The image and labels are prepared before any method is run. This prevents the resolver comparison from being confounded by different rescaling or normalization.

In [ ]:
AUGMENTER = Augmentations()
TARGET_NAMES = ('nuclei', 'cells')

def _labels_from_item(item):
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    reference = _resolve_array(nucleus if nucleus is not None else cell)
    nucleus_array = np.full(reference.shape, -1, dtype=np.int32) if nucleus is None else _resolve_array(nucleus).astype(np.int32, copy=False)
    cell_array = np.full(reference.shape, -1, dtype=np.int32) if cell is None else _resolve_array(cell).astype(np.int32, copy=False)
    if nucleus_array.shape != cell_array.shape:
        raise ValueError(f'Label shape mismatch for {item.get("filename", "<unnamed>")}')
    return np.stack((nucleus_array, cell_array), axis=0)

def prepare_record(item):
    image = _resolve_array(item['image'])
    labels = _labels_from_item(item)
    image_tensor, labels_tensor = AUGMENTER.to_tensor(image, labels, normalize=False)
    image_tensor, _ = AUGMENTER.normalize(image_tensor)
    image_tensor, labels_tensor = AUGMENTER.torch_rescale(
        image_tensor, labels_tensor,
        current_pixel_size=float(item['pixel_size']),
        requested_pixel_size=MODEL_PIXEL_SIZE_UM,
        crop=False, modality='Fluorescence',
    )
    return image_tensor.contiguous().float(), labels_tensor.contiguous().to(torch.int32)

example_info, example_item = RECORDS[0]
example_image, example_labels = prepare_record(example_item)
print({
    'record_id': example_info['record_id'],
    'prepared_image': tuple(example_image.shape),
    'prepared_labels': tuple(example_labels.shape),
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
})

## 4. Controlled public tiled inference

The public model is evaluated through its raw network and Python postprocessor. This avoids the sparse TorchScript postprocessor failure seen in earlier CUDA/CPU comparisons while retaining the public model's postprocessing behavior. The safe sparse functions are patched only during each postprocessing call and restored afterward.

In [ ]:
ORIGINAL_FAST_SPARSE_IOU = loss_utils.fast_sparse_iou
ORIGINAL_FIND_CONNECTED_COMPONENTS = loss_utils.find_connected_components

def _safe_fast_sparse_iou(sparse_onehot):
    sparse_onehot = sparse_onehot.coalesce()
    intersection = torch.sparse.mm(sparse_onehot, sparse_onehot.T).to_dense()
    sizes = torch.sparse.sum(sparse_onehot, dim=(1,))[None].to_dense()
    union = sizes.T + sizes - intersection
    iou = intersection / union
    return (iou + iou.T) * 0.5

def _safe_find_connected_components(adjacency_matrix, max_iterations=100):
    symmetric = torch.logical_or(adjacency_matrix != 0, adjacency_matrix.T != 0).to(adjacency_matrix.dtype)
    return ORIGINAL_FIND_CONNECTED_COMPONENTS(symmetric, max_iterations)

@contextmanager
def _safe_public_postprocessing():
    previous_iou = loss_utils.fast_sparse_iou
    previous_components = loss_utils.find_connected_components
    loss_utils.fast_sparse_iou = _safe_fast_sparse_iou
    loss_utils.find_connected_components = _safe_find_connected_components
    try:
        yield
    finally:
        loss_utils.fast_sparse_iou = previous_iou
        loss_utils.find_connected_components = previous_components

class PublicPythonPredictor(nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network
        public_device = str(next(iter(network.parameters())).device)
        self.postprocessor = LossInstanSeg(
            n_sigma=int(network.n_sigma), binary_loss_fn_str='lovasz_hinge', seed_loss_fn='binary_xloss',
            device=public_device, cells_and_nuclei=bool(network.cells_and_nuclei),
            window_size=int(network.default_window_size), dim_coords=int(network.dim_coords),
            dim_seeds=int(network.dim_seeds), feature_engineering_function='0',
        )
        self.postprocessor.feature_engineering = loss_utils.feature_engineering_slow
        self.postprocessor.pixel_classifier = network.pixel_classifier

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        with torch.amp.autocast(image_batch.device.type, enabled=False):
            model_input = image_batch.clamp(min=-2, max=3)
            model_input, padding = _instanseg_padding(model_input, extra_pad=0)
            raw_output = self.network.fcn(model_input)
            raw_output = _recover_padding(raw_output, padding)
            with _safe_public_postprocessing():
                labels = [
                    self.postprocessor.postprocessing(
                        raw_sample, device=image_batch.device,
                        classifier=self.network.pixel_classifier, **kwargs
                    )
                    for raw_sample in raw_output
                ]
            labels = torch.stack(labels)
            if resolve_cell_and_nucleus and labels.shape[1] == 2:
                labels = loss_utils.resolve_cell_and_nucleus_boundaries(labels.float())
            return labels.float()

public_predictor = PublicPythonPredictor(network).eval()

def _pad_for_model(image_tensor):
    original_shape = tuple(image_tensor.shape[-2:])
    pad_height = (-original_shape[0]) % INFERENCE_DIMENSION_MULTIPLE
    pad_width = (-original_shape[1]) % INFERENCE_DIMENSION_MULTIPLE
    if pad_height == 0 and pad_width == 0:
        return image_tensor, original_shape
    mode = 'replicate' if pad_height >= original_shape[0] or pad_width >= original_shape[1] else 'reflect'
    return torch.nn.functional.pad(image_tensor, (0, pad_width, 0, pad_height), mode=mode), original_shape

def predict_public(image_tensor, resolve_cell_and_nucleus):
    model_input, original_shape = _pad_for_model(image_tensor)
    kwargs = dict(POSTPROCESSING)
    kwargs['resolve_cell_and_nucleus'] = bool(resolve_cell_and_nucleus)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input, public_predictor,
            window_size=(INFERENCE_TILE_SIZE, INFERENCE_TILE_SIZE),
            sw_device=DEVICE, device='cpu', batch_size=INFERENCE_BATCH_SIZE,
            output_channels=2, show_progress=False, instanseg_kwargs=kwargs,
        )
    prediction = prediction.squeeze(0).to(torch.int32).cpu()
    prediction = prediction[..., :original_shape[0], :original_shape[1]]
    if tuple(prediction.shape) != (2, *original_shape):
        raise ValueError(f'Unexpected prediction shape {tuple(prediction.shape)} for {original_shape}')
    return prediction.numpy()

print('Controlled public predictor ready.')

## 5. Apply the custom watershed resolver

A nucleus qualifies for a cell when strictly more than 50% of its pixels overlap that cell. Only cells with multiple qualifying nuclei are divided. Unnucleated cells and singly nucleated cells are preserved. Unmatched nuclei are not converted into proxy cells in this F1 test, so the watershed branch does not invent additional objects.

In [ ]:
def associate_nuclei_to_cells(cell_labels, nucleus_labels, min_overlap_fraction=WATERSHED_OVERLAP_FRACTION):
    both = (cell_labels > 0) & (nucleus_labels > 0)
    columns = ['cell_id', 'nucleus_id', 'overlap_pixels', 'nucleus_area', 'overlap_fraction']
    if not np.any(both):
        return pd.DataFrame(columns=columns)
    cell_ids = cell_labels[both].astype(np.int64, copy=False)
    nucleus_ids = nucleus_labels[both].astype(np.int64, copy=False)
    pair_shape = (int(cell_labels.max()) + 1, int(nucleus_labels.max()) + 1)
    pair_codes = np.ravel_multi_index((cell_ids, nucleus_ids), pair_shape)
    unique_codes, overlap_pixels = np.unique(pair_codes, return_counts=True)
    pair_cell_ids, pair_nucleus_ids = np.unravel_index(unique_codes, pair_shape)
    positive_nuclei = nucleus_labels[nucleus_labels > 0]
    nucleus_ids_all, nucleus_areas = np.unique(positive_nuclei, return_counts=True)
    area_by_id = pd.Series(nucleus_areas, index=nucleus_ids_all)
    table = pd.DataFrame({
        'cell_id': pair_cell_ids.astype(np.uint32),
        'nucleus_id': pair_nucleus_ids.astype(np.uint32),
        'overlap_pixels': overlap_pixels,
    })
    table['nucleus_area'] = table['nucleus_id'].map(area_by_id).astype(np.int64)
    table['overlap_fraction'] = table['overlap_pixels'] / table['nucleus_area']
    return table.loc[table['overlap_fraction'] > min_overlap_fraction, columns].copy()

def _combine_slices(slices, margin, shape):
    y0 = max(0, min(item[0].start for item in slices) - margin)
    y1 = min(shape[0], max(item[0].stop for item in slices) + margin)
    x0 = max(0, min(item[1].start for item in slices) - margin)
    x1 = min(shape[1], max(item[1].stop for item in slices) + margin)
    return slice(y0, y1), slice(x0, x1)

def apply_watershed_resolve(raw_cells, raw_nuclei):
    raw_cell_ids = np.unique(raw_cells[raw_cells > 0]).astype(np.uint32)
    qualifying_pairs = associate_nuclei_to_cells(raw_cells, raw_nuclei)
    qualifying_by_cell = {
        int(cell_id): group['nucleus_id'].astype(int).tolist()
        for cell_id, group in qualifying_pairs.groupby('cell_id', sort=True)
    }
    cell_slices = {int(label_id): bbox for label_id, bbox in enumerate(ndi.find_objects(raw_cells), start=1) if bbox is not None}
    nucleus_slices = {int(label_id): bbox for label_id, bbox in enumerate(ndi.find_objects(raw_nuclei), start=1) if bbox is not None}
    output = raw_cells.copy()
    next_cell_id = int(raw_cells.max()) + 1
    nucleus_to_final_cell = {}
    records = []

    for cell_id in raw_cell_ids.astype(int):
        associated = qualifying_by_cell.get(cell_id, [])
        if len(associated) == 1:
            nucleus_to_final_cell[associated[0]] = cell_id
            continue
        if len(associated) <= 1 or cell_id not in cell_slices or not all(nucleus_id in nucleus_slices for nucleus_id in associated):
            continue

        bbox = _combine_slices(
            [cell_slices[cell_id]] + [nucleus_slices[nucleus_id] for nucleus_id in associated],
            margin=8, shape=raw_cells.shape,
        )
        cell_crop = raw_cells[bbox]
        nucleus_crop = raw_nuclei[bbox]
        territory = cell_crop == cell_id
        markers = np.zeros(territory.shape, dtype=np.int32)
        for marker_id, nucleus_id in enumerate(associated, start=1):
            nucleus_mask = nucleus_crop == nucleus_id
            territory |= nucleus_mask
            markers[nucleus_mask] = marker_id
        split = watershed(
            -ndi.distance_transform_edt(territory),
            markers=markers, mask=territory,
        )
        daughter_ids = [cell_id] + list(range(next_cell_id, next_cell_id + len(associated) - 1))
        next_cell_id += len(associated) - 1
        local_output = output[bbox]
        parent_pixels = cell_crop == cell_id
        local_output[parent_pixels] = 0
        for marker_id, (nucleus_id, daughter_id) in enumerate(zip(associated, daughter_ids), start=1):
            local_output[(split == marker_id) & parent_pixels] = daughter_id
            nucleus_to_final_cell[nucleus_id] = daughter_id
        output[bbox] = local_output
        records.append({
            'parent_cell_id': cell_id,
            'n_nuclei': len(associated),
            'n_daughters': len(daughter_ids),
        })

    # Put qualifying nuclei back into their associated cell/daughter territories.
    for nucleus_id, final_cell_id in nucleus_to_final_cell.items():
        output[raw_nuclei == nucleus_id] = final_cell_id
    return output, qualifying_pairs, pd.DataFrame(records)

def count_instances(labels):
    return int(np.unique(labels[labels > 0]).size)

## 6. Run the paired F1 evaluation

In [ ]:
def _evaluation_arrays(gt, prediction, target_index):
    gt_eval = torch.as_tensor(gt[target_index], dtype=torch.int32).clone()
    prediction_eval = torch.as_tensor(prediction[target_index], dtype=torch.int32).clone()
    valid = gt_eval >= 0
    gt_eval[~valid] = 0
    prediction_eval[~valid] = 0
    if not bool((gt_eval > 0).any()):
        return None
    return gt_eval, prediction_eval

def metric_rows(evaluation_set, info, ground_truth, predictions):
    rows = []
    for method, prediction in predictions.items():
        for target_index, target_name in enumerate(TARGET_NAMES):
            arrays = _evaluation_arrays(ground_truth, prediction, target_index)
            if arrays is None:
                continue
            gt_eval, prediction_eval = arrays
            for threshold, stat in zip(THRESHOLDS, matching_torch(gt_eval, prediction_eval, THRESHOLDS)):
                rows.append({
                    'evaluation_set': evaluation_set,
                    'method': method,
                    'method_label': METHOD_LABELS[method],
                    'target': target_name,
                    'threshold': float(threshold),
                    'record_id': info['record_id'],
                    'filename': info['filename'],
                    'parent_dataset': info['parent_dataset'],
                    'tp': int(stat.tp), 'fp': int(stat.fp), 'fn': int(stat.fn),
                    'precision': float(stat.precision), 'recall': float(stat.recall),
                    'f1': float(stat.f1),
                    'n_true': int(stat.n_true), 'n_pred': int(stat.n_pred),
                })
    return rows

def aggregate_metrics(per_record):
    if per_record.empty:
        return pd.DataFrame(), pd.DataFrame()
    threshold_rows = []
    group_columns = ['evaluation_set', 'method', 'method_label', 'target', 'threshold']
    for keys, group in per_record.groupby(group_columns, sort=True):
        evaluation_set, method, method_label, target, threshold = keys
        tp, fp, fn = (int(group[name].sum()) for name in ('tp', 'fp', 'fn'))
        threshold_rows.append({
            'evaluation_set': evaluation_set, 'method': method, 'method_label': method_label,
            'target': target, 'threshold': threshold, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': tp / max(tp + fp, 1e-10),
            'recall': tp / max(tp + fn, 1e-10),
            'f1': 2 * tp / max(2 * tp + fp + fn, 1e-10),
            'macro_f1': float(group['f1'].mean()),
            'n_true': int(group['n_true'].sum()), 'n_pred': int(group['n_pred'].sum()),
            'n_records': int(group['record_id'].nunique()),
        })
    threshold_table = pd.DataFrame(threshold_rows)
    summary = threshold_table.loc[threshold_table['threshold'] == 0.5].copy()
    return summary, threshold_table

PER_RECORD_DF = pd.DataFrame()
SUMMARY_DF = pd.DataFrame()
THRESHOLD_DF = pd.DataFrame()
DIAGNOSTICS_DF = pd.DataFrame()
SPOT_CHECKS = {}

if not RUN_EVALUATION:
    print('Evaluation is gated. Set RUN_EVALUATION=True on a CUDA kernel.')
else:
    metric_rows_all = []
    diagnostic_rows = []
    started = time.perf_counter()
    for evaluation_set, records in RECORD_GROUPS.items():
        print(f'[{evaluation_set}] {len(records)} records', flush=True)
        for number, (info, item) in enumerate(records, start=1):
            image_tensor, ground_truth = prepare_record(item)
            unresolved = predict_public(image_tensor, resolve_cell_and_nucleus=False)
            native = predict_public(image_tensor, resolve_cell_and_nucleus=True)
            watershed_cells, qualifying_pairs, watershed_records = apply_watershed_resolve(
                unresolved[1], unresolved[0]
            )
            predictions = {
                'unresolved': unresolved,
                'native': native,
                'watershed': np.stack((unresolved[0], watershed_cells), axis=0),
            }
            metric_rows_all.extend(metric_rows(evaluation_set, info, ground_truth.numpy(), predictions))
            diagnostic_rows.extend([
                {'evaluation_set': evaluation_set, 'record_id': info['record_id'], 'method': 'unresolved',
                 'nuclei': count_instances(unresolved[0]), 'cells': count_instances(unresolved[1]),
                 'qualifying_pairs': int(len(qualifying_pairs)), 'ambiguous_parent_cells': int((qualifying_pairs.groupby('cell_id').size() > 1).sum()) if not qualifying_pairs.empty else 0,
                 'watershed_splits': 0, 'changed_cell_pixels': 0},
                {'evaluation_set': evaluation_set, 'record_id': info['record_id'], 'method': 'native',
                 'nuclei': count_instances(native[0]), 'cells': count_instances(native[1]),
                 'qualifying_pairs': np.nan, 'ambiguous_parent_cells': np.nan,
                 'watershed_splits': np.nan, 'changed_cell_pixels': np.nan},
                {'evaluation_set': evaluation_set, 'record_id': info['record_id'], 'method': 'watershed',
                 'nuclei': count_instances(unresolved[0]), 'cells': count_instances(watershed_cells),
                 'qualifying_pairs': int(len(qualifying_pairs)), 'ambiguous_parent_cells': int((qualifying_pairs.groupby('cell_id').size() > 1).sum()) if not qualifying_pairs.empty else 0,
                 'watershed_splits': int(len(watershed_records)),
                 'changed_cell_pixels': int(np.count_nonzero(watershed_cells != unresolved[1]))},
            ])
            if number <= SPOT_CHECK_RECORDS:
                SPOT_CHECKS[(evaluation_set, info['record_id'])] = {
                    'image': image_tensor.cpu(), 'ground_truth': ground_truth.cpu(),
                    'predictions': predictions, 'info': info,
                }
            if number == 1 or number % 10 == 0 or number == len(records):
                print(f'  {number}/{len(records)}', flush=True)
            del image_tensor, ground_truth, unresolved, native, watershed_cells, predictions
            gc.collect()
            torch.cuda.empty_cache()
    PER_RECORD_DF = pd.DataFrame(metric_rows_all)
    DIAGNOSTICS_DF = pd.DataFrame(diagnostic_rows)
    SUMMARY_DF, THRESHOLD_DF = aggregate_metrics(PER_RECORD_DF)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes.')
    display(SUMMARY_DF[['evaluation_set', 'method', 'target', 'n_records', 'n_true', 'n_pred', 'precision', 'recall', 'f1', 'macro_f1']])
    if SAVE_ARTIFACTS:
        RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
        PER_RECORD_DF.to_csv(RESULTS_ROOT / 'per_record_metrics.csv', index=False)
        SUMMARY_DF.to_csv(RESULTS_ROOT / 'summary_iou50.csv', index=False)
        THRESHOLD_DF.to_csv(RESULTS_ROOT / 'threshold_metrics.csv', index=False)
        DIAGNOSTICS_DF.to_csv(RESULTS_ROOT / 'postprocessing_diagnostics.csv', index=False)
        provenance = {
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'dataset_file': str(DATASET_FILE), 'source_root': str(SOURCE_ROOT),
            'model': PUBLIC_MODEL_NAME, 'model_version': PUBLIC_MODEL_VERSION,
            'model_pixel_size_um': MODEL_PIXEL_SIZE_UM, 'tile_size_px': INFERENCE_TILE_SIZE,
            'postprocessing': POSTPROCESSING, 'thresholds': THRESHOLDS,
            'methods': METHOD_LABELS,
            'watershed_rule': 'strictly greater than 0.5 nucleus-pixel overlap; local distance-transform watershed for multiply qualifying cells; no proxy cells',
            'record_ids': [info['record_id'] for info, _ in RECORDS],
        }
        (RESULTS_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2, default=str) + '\n')
        print('Saved artifacts:', RESULTS_ROOT)

## 7. F1 results and paired watershed deltas

The pooled F1 sums true positives, false positives, and false negatives across records. Macro F1 gives every record equal weight. The paired table compares methods on exactly the same record and target, which is the most direct measure of whether watershed changed the score.

In [ ]:
if SUMMARY_DF.empty and (RESULTS_ROOT / 'summary_iou50.csv').is_file():
    SUMMARY_DF = pd.read_csv(RESULTS_ROOT / 'summary_iou50.csv')
if THRESHOLD_DF.empty and (RESULTS_ROOT / 'threshold_metrics.csv').is_file():
    THRESHOLD_DF = pd.read_csv(RESULTS_ROOT / 'threshold_metrics.csv')
if PER_RECORD_DF.empty and (RESULTS_ROOT / 'per_record_metrics.csv').is_file():
    PER_RECORD_DF = pd.read_csv(RESULTS_ROOT / 'per_record_metrics.csv')
if not SUMMARY_DF.empty:
    display(SUMMARY_DF[['evaluation_set', 'method_label', 'target', 'n_records', 'n_true', 'n_pred', 'precision', 'recall', 'f1', 'macro_f1']])

if not THRESHOLD_DF.empty:
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for axis, target in zip(axes, TARGET_NAMES):
        data = THRESHOLD_DF[THRESHOLD_DF['target'] == target]
        for (evaluation_set, method_label), curve in data.groupby(['evaluation_set', 'method_label']):
            axis.plot(curve['threshold'], curve['f1'], marker='o', label=f'{evaluation_set}: {method_label}')
        axis.set_title(target.title())
        axis.set_xlabel('IoU threshold')
        axis.set_ylabel('Pooled F1')
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.25)
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    figure.tight_layout()
    plt.show()

    # THRESHOLD_DF is pooled by method/target/threshold and intentionally has
    # no record_id. Use the long per-record table for paired deltas.
    if PER_RECORD_DF.empty or 'record_id' not in PER_RECORD_DF.columns:
        print('Per-record metrics are unavailable; rerun the evaluation cell or load per_record_metrics.csv.')
    else:
        at_05 = PER_RECORD_DF[PER_RECORD_DF['threshold'] == 0.5].copy()
        paired = at_05.pivot_table(
            index=['evaluation_set', 'record_id', 'target'],
            columns='method', values='f1', aggfunc='first',
        ).reset_index()
        for method in METHODS:
            if method not in paired.columns:
                paired[method] = np.nan
        paired['watershed_minus_unresolved'] = paired['watershed'] - paired['unresolved']
        paired['watershed_minus_native'] = paired['watershed'] - paired['native']
        display(paired.groupby(['evaluation_set', 'target'])[['unresolved', 'native', 'watershed', 'watershed_minus_unresolved', 'watershed_minus_native']].agg(['mean', 'median', 'count']))
        if SAVE_ARTIFACTS:
            paired.to_csv(RESULTS_ROOT / 'paired_iou50_deltas.csv', index=False)

## 8. Optional visual spot check

This is only a sanity check that the F1 change corresponds to the intended cell-boundary changes. The watershed branch keeps the unresolved nuclear mask unchanged.

In [ ]:
def _display_base(image_tensor):
    base = np.asarray(image_tensor).mean(axis=0)
    low, high = np.percentile(base, (1, 99))
    return np.clip((base - low) / max(float(high - low), 1e-6), 0, 1)

def _show_mask_overlay(axis, base, labels, title):
    axis.imshow(base, cmap='gray', interpolation='nearest')
    axis.contour(find_boundaries(labels[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.45)
    axis.contour(find_boundaries(labels[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.45)
    axis.set_title(title, fontsize=9)
    axis.axis('off')

for (evaluation_set, record_id), entry in SPOT_CHECKS.items():
    base = _display_base(entry['image'].numpy())
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(base, cmap='gray'); axes[0].set_title('Input'); axes[0].axis('off')
    _show_mask_overlay(axes[1], base, entry['ground_truth'].numpy(), 'Ground truth')
    for axis, method in zip(axes[2:], ('unresolved', 'native')):
        _show_mask_overlay(axis, base, entry['predictions'][method], METHOD_LABELS[method])
    fig, axis = plt.subplots(figsize=(5, 5))
    _show_mask_overlay(axis, base, entry['predictions']['watershed'], METHOD_LABELS['watershed'])
    fig.suptitle(f'{evaluation_set} | {record_id}', y=1.02)
    plt.show()

## Interpretation

- The primary comparison is **cell F1**: `watershed` versus `unresolved` measures the effect of the custom split, while `watershed` versus `native` compares two different reconciliation policies.
- Nuclear F1 for `unresolved` and `watershed` should be identical because watershed changes only the cell plane. A difference indicates an implementation or metric-selection problem.
- A lower cell F1 after watershed means that the postprocessing changed object matching unfavorably, even if the masks look more filled. A higher hole count or lower hole count alone is not an accuracy metric.
- If the watershed gain is confined to a few records, inspect the paired deltas and spot-check those records before adopting it.
- This notebook does not create proxy cells for unmatched nuclei and does not overwrite canonical pipeline masks.